In [1]:
import requests
from bs4 import BeautifulSoup

url = "https://oilprice.com/Latest-Energy-News/World-News/Page-812.html"
resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(resp.text, "html.parser")

for item in soup.select("div.categoryArticle"):
    title_el = item.select_one("h2.categoryArticle__title")
    meta_el = item.select_one("p.categoryArticle__meta")
    if title_el:
        print(title_el.get_text(strip=True), "|", meta_el.get_text(strip=True) if meta_el else None)

Shareholders Want Barclays To Quit Funding Fossil Fuel Companies | Jan 08, 2020 at 11:01 | Irina Slav
Saudi Tanker Group Halts Strait Of Hormuz Route | Jan 08, 2020 at 10:31 | Tsvetana Paraskova
Iraq Knew The U.S. Attack Was Going To Happen | Jan 08, 2020 at 09:10 | Tsvetana Paraskova
Oil Prices Soar As Iran Fires Missiles At U.S. Base | Jan 07, 2020 at 18:07 | Tom Kool
Soaring Gasoline, Distillate Inventories Offset Large Crude Draw | Jan 07, 2020 at 15:50 | Julianne Geiger
U.S. Oil Boom Suppressed Oil Prices In 2019 | Jan 07, 2020 at 15:00 | Tsvetana Paraskova
U.S. Trade Deficit At Three-Year Low As Oil Imports Dip | Jan 07, 2020 at 13:24 | Tsvetana Paraskova
Libya Blames Low Oil Prices For Tanking Revenue | Jan 07, 2020 at 12:28 | Tsvetana Paraskova
Work On Controversial Coastal GasLink Pipeline To Continue | Jan 07, 2020 at 12:20 | Irina Slav
Massive Strikes In France Block Exxon Oil Refinery | Jan 07, 2020 at 09:47 | Tsvetana Paraskova
Mexican President Decides To Re-Route Crucial

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from pathlib import Path

Path("scraped").mkdir(exist_ok=True)
HEADERS = {"User-Agent": "Mozilla/5.0"}


def scrape_page(page_num, session):
    if page_num == 1:
        url = "https://oilprice.com/Latest-Energy-News/World-News/"
    else:
        url = f"https://oilprice.com/Latest-Energy-News/World-News/Page-{page_num}.html"

    resp = session.get(url, headers=HEADERS, timeout=20)

    if resp.status_code != 200:
        # request itself failed (blocked, rate-limited, page missing) -- signal this
        # distinctly from "page loaded fine but had zero articles"
        return None, resp.status_code

    soup = BeautifulSoup(resp.text, "html.parser")
    rows = []
    for item in soup.select("div.categoryArticle"):
        title_el = item.select_one("h2.categoryArticle__title")
        meta_el = item.select_one("p.categoryArticle__meta")
        if title_el:
            rows.append({
                "title": title_el.get_text(strip=True),
                "meta": meta_el.get_text(strip=True) if meta_el else None,
                "page": page_num
            })
    return rows, 200


def page_already_done(out_path):
    # a checkpoint only counts as "done" if the file exists AND contains rows
    if not out_path.exists():
        return False
    try:
        existing = pd.read_csv(out_path)
        return len(existing) > 0
    except pd.errors.EmptyDataError:
        return False


session = requests.Session()
failed_pages = []

for page in range(1, 813):
    out_path = Path(f"scraped/page_{page:04d}.csv")

    if page_already_done(out_path):
        continue

    rows, status = scrape_page(page, session)

    if rows is None:
        print(f"Page {page}: request failed (HTTP {status}) -- will retry next run")
        failed_pages.append(page)
        time.sleep(5)  # back off longer after a failure
        continue

    if len(rows) == 0:
        print(f"Page {page}: loaded fine but 0 articles found -- selector may need checking")

    pd.DataFrame(rows).to_csv(out_path, index=False)

    if page % 50 == 0:
        print(f"{page}/812 done")

    time.sleep(2)

print(f"\nFinished this run. {len(failed_pages)} pages failed and will be retried: {failed_pages}")

# combine only checkpoint files that actually contain data
csv_files = list(Path("scraped").glob("page_*.csv"))
print(f"Found {len(csv_files)} checkpoint files on disk")

dfs = []
for f in csv_files:
    try:
        df = pd.read_csv(f)
        if len(df) > 0:
            dfs.append(df)
    except pd.errors.EmptyDataError:
        continue

combined = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
combined.to_csv("oilprice_headlines_2020_2026.csv", index=False)
print(f"Combined shape: {combined.shape}")

100/812 done
150/812 done
200/812 done
250/812 done
300/812 done
350/812 done
400/812 done
450/812 done
500/812 done
550/812 done
600/812 done
650/812 done
700/812 done
750/812 done
800/812 done

Finished this run. 0 pages failed and will be retried: []
Found 812 checkpoint files on disk
Combined shape: (16240, 3)


In [3]:
!pip install cloudscraper --user

Access is denied.


In [5]:
import sys
!{sys.executable} -m pip install cloudscraper

Defaulting to user installation because normal site-packages is not writeable


In [7]:
import cloudscraper
from bs4 import BeautifulSoup
import pandas as pd
import time
from pathlib import Path

Path("scraped_investing").mkdir(exist_ok=True)

BASE = "https://www.investing.com/news/commodities-news"
TOTAL_PAGES = 286


def scrape_page(page_num, scraper):
    if page_num == 1:
        url = BASE
    else:
        url = f"{BASE}/{page_num}"

    resp = scraper.get(url, timeout=20)

    if resp.status_code != 200:
        return None, resp.status_code

    soup = BeautifulSoup(resp.text, "html.parser")

    rows = []
    articles = soup.select("article[data-test='article-item']") or soup.select("article")

    for item in articles:
        title_el = (item.select_one("a[data-test='article-title-link']")
                    or item.select_one("h3 a")
                    or item.select_one("a"))
        desc_el = item.select_one("p")
        author_el = item.select_one("[data-test='news-provider-name']")
        time_el = item.select_one("time") or item.select_one("[data-test='article-publish-date']")

        title = title_el.get_text(strip=True) if title_el else None
        link = title_el.get("href") if title_el else None
        description = desc_el.get_text(strip=True) if desc_el else None
        author = author_el.get_text(strip=True) if author_el else None
        date = time_el.get("datetime") if time_el else (time_el.get_text(strip=True) if time_el else None)

        if title:
            rows.append({
                "title": title,
                "description": description,
                "author": author,
                "date": date,
                "link": link,
                "page": page_num
            })

    return rows, 200


def page_already_done(out_path):
    if not out_path.exists():
        return False
    try:
        existing = pd.read_csv(out_path)
        return len(existing) > 0
    except pd.errors.EmptyDataError:
        return False


scraper = cloudscraper.create_scraper(
    browser={"browser": "chrome", "platform": "windows", "mobile": False}
)
failed_pages = []

for page in range(1, TOTAL_PAGES + 1):
    out_path = Path(f"scraped_investing/page_{page:04d}.csv")

    if page_already_done(out_path):
        continue

    rows, status = scrape_page(page, scraper)

    if rows is None:
        print(f"Page {page}: request failed (HTTP {status}) -- will retry next run")
        failed_pages.append(page)
        time.sleep(5)
        continue

    if len(rows) == 0:
        print(f"Page {page}: loaded fine but 0 articles found -- may need selector check")

    pd.DataFrame(rows).to_csv(out_path, index=False)

    if page % 20 == 0:
        print(f"{page}/{TOTAL_PAGES} done")

    time.sleep(2)

print(f"\nFinished this run. {len(failed_pages)} pages failed and will be retried: {failed_pages}")

csv_files = list(Path("scraped_investing").glob("page_*.csv"))
print(f"Found {len(csv_files)} checkpoint files on disk")

dfs = []
for f in csv_files:
    try:
        df = pd.read_csv(f)
        if len(df) > 0:
            dfs.append(df)
    except pd.errors.EmptyDataError:
        continue

combined = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
combined.to_csv("investing_commodities_news.csv", index=False)
print(f"Combined shape: {combined.shape}")

20/286 done
40/286 done
60/286 done
80/286 done
100/286 done
120/286 done
140/286 done
160/286 done
180/286 done
200/286 done
220/286 done
240/286 done
260/286 done
280/286 done

Finished this run. 0 pages failed and will be retried: []
Found 286 checkpoint files on disk
Combined shape: (10010, 6)


In [14]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from pathlib import Path

Path("scraped_hellenic").mkdir(exist_ok=True)
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

BASE = "https://www.hellenicshippingnews.com/category/oil-energy/oil-companies-news"
TOTAL_PAGES = 540


def scrape_page(page_num, session):
    if page_num == 1:
        url = f"{BASE}/"
    else:
        url = f"{BASE}/page/{page_num}/"

    resp = session.get(url, headers=HEADERS, timeout=20)

    if resp.status_code != 200:
        return None, resp.status_code

    soup = BeautifulSoup(resp.text, "html.parser")

   
session = requests.Session()
failed_pages = []

for page in range(1, TOTAL_PAGES + 1):
    out_path = Path(f"scraped_hellenic/page_{page:04d}.csv")

    if page_already_done(out_path):
        continue

    rows, status = scrape_page(page, session)

    if rows is None:
        print(f"Page {page}: request failed (HTTP {status}) -- will retry next run")
        failed_pages.append(page)
        time.sleep(5)
        continue

    if len(rows) == 0:
        print(f"Page {page}: loaded fine but 0 articles found -- selector may need checking")

    pd.DataFrame(rows).to_csv(out_path, index=False)

    if page % 30 == 0:
        print(f"{page}/{TOTAL_PAGES} done")

    time.sleep(1.5)

print(f"\nFinished this run. {len(failed_pages)} pages failed and will be retried: {failed_pages}")

csv_files = list(Path("scraped_hellenic").glob("page_*.csv"))
print(f"Found {len(csv_files)} checkpoint files on disk")

dfs = []
for f in csv_files:
    try:
        df = pd.read_csv(f)
        if len(df) > 0:
            dfs.append(df)
    except pd.errors.EmptyDataError:
        continue

combined = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
combined.to_csv("hellenic_oil_news.csv", index=False)
print(f"Combined shape: {combined.shape}")


Finished this run. 0 pages failed and will be retried: []
Found 540 checkpoint files on disk
Combined shape: (8100, 4)


In [13]:
import pandas as pd
import re
from datetime import datetime
from difflib import SequenceMatcher

# ---------- Load ----------
oilprice = pd.read_csv("oilprice_headlines_2020_2026.csv")
investing = pd.read_csv("investing_commodities_news.csv")
hellenic = pd.read_csv("hellenic_oil_news.csv")

print(f"Raw counts -- oilprice: {len(oilprice)}, investing: {len(investing)}, hellenic: {len(hellenic)}")

# ---------- Normalize each source to: date, title, source, author ----------

def parse_oilprice_meta(meta):
    if pd.isna(meta):
        return None, None
    parts = str(meta).split("|")
    date_str = parts[0].strip() if len(parts) > 0 else None
    author = parts[1].strip() if len(parts) > 1 else None
    parsed_date = None
    if date_str:
        date_only = date_str.split(" at ")[0].strip()
        try:
            parsed_date = datetime.strptime(date_only, "%b %d, %Y").date()
        except ValueError:
            parsed_date = None
    return parsed_date, author

oilprice[["date", "author"]] = oilprice["meta"].apply(
    lambda m: pd.Series(parse_oilprice_meta(m))
)
oilprice["source"] = "oilprice"
oilprice = oilprice[["date", "title", "source", "author"]]

investing["date"] = pd.to_datetime(investing["date"], errors="coerce").dt.date
investing["source"] = "investing"
investing = investing[["date", "title", "source", "author"]]

hellenic["date"] = pd.to_datetime(hellenic["date"], format="%d/%m/%Y", errors="coerce").dt.date
hellenic["source"] = "hellenic"
hellenic["author"] = None
hellenic = hellenic[["date", "title", "source", "author"]]

# ---------- Merge ----------
combined = pd.concat([oilprice, investing, hellenic], ignore_index=True)

before_na_drop = len(combined)
combined = combined.dropna(subset=["date", "title"])
print(f"\nDropped {before_na_drop - len(combined)} rows with missing date/title")
print(f"Before dedup: {len(combined)} headlines")

# ---------- Deduplicate ----------
combined = combined.drop_duplicates(subset=["date", "title"])
print(f"After exact-title dedup: {len(combined)} headlines")


def normalize_title(t):
    t = t.lower()
    t = re.sub(r"[^a-z0-9\s]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def similar(a, b):
    return SequenceMatcher(None, a, b).ratio()


combined["_norm_title"] = combined["title"].apply(normalize_title)

kept_rows = []
for date, group in combined.groupby("date"):
    group = group.reset_index(drop=True)
    keep_mask = [True] * len(group)

    for i in range(len(group)):
        if not keep_mask[i]:
            continue
        for j in range(i + 1, len(group)):
            if not keep_mask[j]:
                continue
            if similar(group.loc[i, "_norm_title"], group.loc[j, "_norm_title"]) > 0.85:
                keep_mask[j] = False

    kept_rows.append(group[keep_mask])

deduped = pd.concat(kept_rows, ignore_index=True).drop(columns=["_norm_title"])
print(f"After fuzzy same-day dedup: {len(deduped)} headlines")

# ---------- Filter to project date range: 2020-01-01 to 2026-07-03 ----------
start_date = pd.to_datetime("2020-01-01").date()
end_date = pd.to_datetime("2026-07-03").date()

before_date_filter = len(deduped)
deduped = deduped[(deduped["date"] >= start_date) & (deduped["date"] <= end_date)]
print(f"\nDropped {before_date_filter - len(deduped)} rows outside 2020-01-01 to 2026-07-03")

deduped = deduped.sort_values("date").reset_index(drop=True)
deduped.to_csv("merged_headlines_deduped.csv", index=False)

# ---------- Daily aggregation prep ----------
daily_counts = deduped.groupby(["date", "source"]).size().unstack(fill_value=0)
daily_counts["total_headlines"] = daily_counts.sum(axis=1)
daily_counts.to_csv("daily_headline_counts.csv")

# ---------- Summary ----------
print(f"\nFinal date range: {deduped['date'].min()} to {deduped['date'].max()}")
print(f"Unique days with coverage: {deduped['date'].nunique()}")
print(f"\nSource breakdown:")
print(deduped["source"].value_counts())
print(f"\nAvg headlines/day: {len(deduped) / deduped['date'].nunique():.1f}")

Raw counts -- oilprice: 16240, investing: 10010, hellenic: 8100

Dropped 0 rows with missing date/title
Before dedup: 34350 headlines
After exact-title dedup: 34300 headlines
After fuzzy same-day dedup: 34272 headlines

Dropped 3 rows outside 2020-01-01 to 2026-07-03

Final date range: 2020-01-03 to 2026-07-03
Unique days with coverage: 2068

Source breakdown:
source
oilprice     16227
investing     9990
hellenic      8052
Name: count, dtype: int64

Avg headlines/day: 16.6
